<a href="https://colab.research.google.com/github/leeet1004835/colab/blob/main/0429.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

分類模型

「梯度遞減」(Gradient descent):
基本概念是先隨機初始化一組係數向量，以迭代更新該組係數向量，一直到J(w)收斂到局部最小值為止。

梯度遞減如何「有方向性地」更新係數向量:
依據損失函數J(w)關於係數向量w的偏微分來決定更新的方向性。
更新幅度則由一個大於零、稱為「學習速率」的常數α決定。

梯度遞減演算方法:
將目前的w0減去學習速率α乘上J(w)關於w0的偏微分、將目前的w1減去學習速率α乘上J(w)
關於w1的偏微分。

J(w)關於w的偏微分就是演算方法中所謂的「梯度」(Gradient):
在迭代過程中w更新的方向性取決於梯度正負號，如果梯度為正，w會向左更新(減小)；如果梯度為負，w會向右更新(增大)。

In [10]:
class GradientDescent:
    """
    This class defines the vanilla gradient descent algorithm for linear regression.
    Args:
        fit_intercept (bool): Whether to add intercept for this model.
    """
    def __init__(self, fit_intercept=True):
        self._fit_intercept = fit_intercept
    def find_gradient(self):
            """
            This function returns the gradient given certain model weights.
            """
            y_hat = np.dot(self._X_train, self._w)
            gradient = (2/self._m) * np.dot(self._X_train.T, y_hat - self._y_train)
            return gradient
    def mean_squared_error(self):
            """
            This function returns the mean squared error given certain model weights.
            """
            y_hat = np.dot(self._X_train, self._w)
            mse = ((y_hat - self._y_train).T.dot(y_hat - self._y_train)) / self._m
            return mse
    def fit(self, X_train, y_train, epochs=10000, learning_rate=0.001):
            """
            This function uses vanilla gradient descent to solve for weights of this model.
            Args:
                X_train (ndarray): 2d-array for feature matrix of training data.
                y_train (ndarray): 1d-array for target vector of training data.
                epochs (int): The number of iterations to update the model weights.
                learning_rate (float): The learning rate of gradient descent.
            """
            self._X_train = X_train.copy()
            self._y_train = y_train.copy()
            self._m = self._X_train.shape[0]
            if self._fit_intercept:
                X0 = np.ones((self._m, 1), dtype=float)
                self._X_train = np.concatenate([X0, self._X_train], axis=1)
            n = self._X_train.shape[1]
            self._w = np.random.rand(n)
            n_prints = 10
            print_iter = epochs // n_prints
            w_history = dict()
            for i in range(epochs):
                current_w = self._w.copy()
                w_history[i] = current_w
                mse = self.mean_squared_error()
                gradient = self.find_gradient()
                if i % print_iter == 0:
                    print("epoch: {:6} - loss: {:.6f}".format(i, mse))
                self._w -= learning_rate*gradient
            w_ravel = self._w.copy().ravel()
            self.intercept_ = w_ravel[0]
            self.coef_ = w_ravel[1:]
            self._w_history = w_history
            return self
    def predict(self, X_test):
            """
            This function returns predicted values with weights of this model.
            Args:
                X_test (ndarray): 2d-array for feature matrix of test data.
            """
            self._X_test = X_test
            m = self._X_test.shape[0]
            if self._fit_intercept:
                X0 = np.ones((m, 1), dtype=float)
                self._X_test = np.concatenate([X0, self._X_test], axis=1)
            y_pred = np.dot(self._X_test, self._w)
            return y_pred

In [11]:
import numpy as np

X0 = np.ones((10, 1))
X1 = np.arange(1, 11).reshape(-1, 1)
w = np.array([5, 6])
X_train = np.concatenate([X0, X1], axis=1)
y_train = np.dot(X_train, w)
print(X_train)
print(y_train)

[[ 1.  1.]
 [ 1.  2.]
 [ 1.  3.]
 [ 1.  4.]
 [ 1.  5.]
 [ 1.  6.]
 [ 1.  7.]
 [ 1.  8.]
 [ 1.  9.]
 [ 1. 10.]]
[11. 17. 23. 29. 35. 41. 47. 53. 59. 65.]


In [12]:
h = GradientDescent(fit_intercept=False)
h.fit(X_train, y_train, epochs=20000, learning_rate=0.001)

epoch:      0 - loss: 1452.863284
epoch:   2000 - loss: 0.500986
epoch:   4000 - loss: 0.093355
epoch:   6000 - loss: 0.017396
epoch:   8000 - loss: 0.003242
epoch:  10000 - loss: 0.000604
epoch:  12000 - loss: 0.000113
epoch:  14000 - loss: 0.000021
epoch:  16000 - loss: 0.000004
epoch:  18000 - loss: 0.000001


In [13]:
print(h.intercept_) # 截距項
print(h.coef_)      # 係數項

4.999204213516389
[6.00011431]


In [15]:
# 將自行定義的梯度遞減預測器類別應用在真實資料
import pandas as pd
from sklearn.model_selection import train_test_split

player_stats = pd.read_csv("https://raw.githubusercontent.com/yaojenkuo/ml-newbies/master/player_stats.csv")
X = player_stats['heightMeters'].values.reshape(-1, 1)
y = player_stats['weightKilograms'].values
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.33, random_state=42)

In [16]:
h = GradientDescent()
h.fit(X_train, y_train, epochs=300000, learning_rate=0.01) # 跑30萬次，學習率(步長)0.01

epoch:      0 - loss: 9294.035384
epoch:  30000 - loss: 53.020356
epoch:  60000 - loss: 49.520124
epoch:  90000 - loss: 48.905110
epoch: 120000 - loss: 48.797048
epoch: 150000 - loss: 48.778061
epoch: 180000 - loss: 48.774725
epoch: 210000 - loss: 48.774139
epoch: 240000 - loss: 48.774036
epoch: 270000 - loss: 48.774017


In [17]:
print(h.intercept_) # 截距項
print(h.coef_)      # 係數項

-95.12931535500506
[97.24445612]


In [18]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
print(lr.intercept_) # 截距項
print(lr.coef_)      # 係數項

-95.14864145823769
[97.25416437]
